# Laboratorio #8

* Josue Say - 228801
* Flavio Galán - 22386

## Repositorio

- [Enlace](https://github.com/JosueSay/labs-ds/tree/main/lab8)
- [Data](https://www.ine.gob.gt/bases-de-datos/accidentes-de-transito/)

## Librerías

In [1]:
import pandas as pd
import pyreadstat
import sys
import os
import re
import glob
import web_scrapping
import subprocess
import unicodedata
import unidecode
import shutil
from contextlib import redirect_stdout
from pyspark.sql import SparkSession, functions as F

## Constantes

In [2]:
def ensureDir(path):
    """Crea un directorio si no existe y avisa si lo crea o si ya existía."""
    if os.path.exists(path):
        if os.path.isdir(path):
            print(f"El directorio ya existe: {path}")
        else:
            # Existe un archivo en la misma ruta, lo eliminamos
            os.remove(path)
            os.makedirs(path, exist_ok=True)
            print(f"Se eliminó un archivo y se creó el directorio: {path}")
    else:
        os.makedirs(path, exist_ok=True)
        print(f"Directorio creado: {path}")

In [3]:
DATA_BASE = "/home/jovyan/work/data"
RAW_BASE = os.path.join(DATA_BASE, "raw")
VARIABLE_BASE = os.path.join(DATA_BASE, "variables")
MANIFEST_BASE = os.path.join(RAW_BASE, "manifest.log")
MANIFEST_BASE_VAR = os.path.join(VARIABLE_BASE, "manifest.log")

# Data que almacena los csv
HECHOS_BASE = os.path.join(DATA_BASE, "hechos")
VEHICULOS_BASE = os.path.join(DATA_BASE, "vehiculos")
FL_BASE = os.path.join(DATA_BASE, "fallecidos_lesionados")

# Caché
CACHE_BASE = "/home/jovyan/work/cache"
CACHE_WEB_SCRAPPING = "cache_web_scrapping.txt"
CACHE_VARIABLES = "cache_variables.txt"
CACHE_CSV_BASE = "cache_csv_base.txt"
CACHE_TRANSFORM_SAV = "cache_transform_sav.txt"
CACHE_SEPARATE = "cache_separate.txt"
CACHE_GENERATED_CSV = "cache_generated_csv.txt"
CACHE_SEPARATE_VAR = os.path.join(CACHE_BASE, "cache_separate_var.txt")

# Preparación archivos unificados
HECHOS_CSV = os.path.join(HECHOS_BASE, "hechos_combinado.csv")
VEHICULOS_CSV = os.path.join(VEHICULOS_BASE, "vehiculos_combinado.csv")
FL_CSV = os.path.join(FL_BASE, "fl_combinado.csv")

HECHOS_COLS = [
    "num_corre", "dia_ocu", "mes_ocu", "dia_sem_ocu", "hora_ocu",
    "g_hora", "depto_ocu", "mupio_ocu", "zona_ocu", "tipo_eve",
    "tipo_veh", "marca_veh", "color_veh", "modelo_veh",
    "anio_ocu_dataset", "anio_ocu_file"
]

VEHICULOS_COLS = [
    "mes_ocu","dia_sem_ocu","hora_ocu","depto_ocu","mupio_ocu","zona_ocu",
    "mayor_menor","tipo_veh","color_veh","modelo_veh","marca_veh","g_edad",
    "anio_ocu_file","tipo_eve","estado_con","g_modelo_veh","sexo_per",
    "edad_per","num_corre","g_hora","anio_ocu_dataset","anio_ocu","edad_grupo"
]

FL_COLS = [
    "num_corre", "dia_ocu", "mes_ocu", "dia_sem_ocu", "hora_ocu",
    "g_hora", "depto_ocu", "mupio_ocu", "zona_ocu",
    "tipo_veh", "marca_veh", "color_veh", "modelo_veh",
    "anio_ocu_dataset", "anio_ocu_file"
]

# Aseguramos que existan las carpetas
ensureDir(DATA_BASE)
ensureDir(CACHE_BASE)
ensureDir(RAW_BASE)
ensureDir(VARIABLE_BASE)
ensureDir(HECHOS_BASE)
ensureDir(VEHICULOS_BASE)
ensureDir(FL_BASE)

El directorio ya existe: /home/jovyan/work/data
El directorio ya existe: /home/jovyan/work/cache
El directorio ya existe: /home/jovyan/work/data/raw
El directorio ya existe: /home/jovyan/work/data/variables
El directorio ya existe: /home/jovyan/work/data/hechos
El directorio ya existe: /home/jovyan/work/data/vehiculos
El directorio ya existe: /home/jovyan/work/data/fallecidos_lesionados


## Obtener data y procesarla

### Ver estructura de carpeta

In [4]:
def printTree(root, prefix=""):
    files = os.listdir(root)
    for i, f in enumerate(files):
        path = os.path.join(root, f)
        connector = "└── " if i == len(files) - 1 else "├── "
        print(prefix + connector + f)
        if os.path.isdir(path):
            extension = "    " if i == len(files) - 1 else "│   "
            printTree(path, prefix + extension)

### Función para ejecutar el script para "web_scrapping.py" obtener data y significado de variables

In [5]:
def runWebScrapping():
    cacheFilePath = os.path.join(CACHE_BASE, CACHE_WEB_SCRAPPING)

    if os.path.exists(cacheFilePath):
        print("Archivo de caché encontrado. Ya se ha ejecutado el proceso de web_scrapping.")
        return

    scriptPath = os.path.join(os.getcwd(), "web_scrapping.py")
    if not os.path.exists(scriptPath):
        print(f"No se encontró {scriptPath}")
        return

    print("Ejecutando web_scrapping.py ...")
    result = subprocess.run(["python", scriptPath], capture_output=True, text=True)

    # Mostrar salida en notebook
    print(result.stdout)
    if result.stderr:
        print("Errores:", result.stderr)

In [6]:
def runWebScrappingVariables():
    cacheFilePath = os.path.join(CACHE_BASE, CACHE_VARIABLES)

    if os.path.exists(cacheFilePath):
        print("Archivo de caché encontrado. Ya se ha ejecutado el proceso de web_scrapping de variables.")
        return

    # Ejecutar ws_variables.py desde el notebook
    scriptPath = os.path.join(os.getcwd(), "ws_variables.py")
    if not os.path.exists(scriptPath):
        print(f"No se encontró {scriptPath}")
        return

    print("Ejecutando ws_variables.py ...")
    result = subprocess.run(["python", scriptPath], capture_output=True, text=True)

    # Mostrar salida en notebook
    print(result.stdout)
    if result.stderr:
        print("Errores:", result.stderr)

In [7]:
def runGbCsv():
    cacheFilePath = os.path.join(CACHE_BASE, CACHE_CSV_BASE)

    if os.path.exists(cacheFilePath):
        print("Archivo de caché encontrado. Ya se ha ejecutado el proceso de gb_csv.")
        return

    # Ejecutar gb_csv.py desde el notebook
    scriptPath = os.path.join(os.getcwd(), "gb_csv.py")
    if not os.path.exists(scriptPath):
        print(f"No se encontró {scriptPath}")
        return

    
    result = subprocess.run(["python", scriptPath], capture_output=True, text=True)
    print("Ejecutando gb_csv.py ...")

    # Mostrar salida en notebook
    print(result.stdout)
    if result.stderr:
        print("Errores:", result.stderr)

### Funciones para pasar archivos `.sav` a formato excel

In [8]:
def convertSavToXlsx(inputPath, outputPath):
    df, meta = pyreadstat.read_sav(inputPath)
    df.to_excel(outputPath, index=False)
    print(f"Convertido: {inputPath} -> {outputPath}")

In [9]:
def processSavFiles():
    cacheFilePath = os.path.join(CACHE_BASE, CACHE_TRANSFORM_SAV)

    if os.path.exists(cacheFilePath):
        print("Archivo de caché encontrado. Ya se ha ejecutado la conversión de .sav a .xlsx.")
        return

    # Leer manifest.log de una vez
    manifest_path = MANIFEST_BASE
    manifest_lines = []
    if os.path.exists(manifest_path):
        with open(manifest_path, "r", encoding="utf-8") as mf:
            manifest_lines = mf.readlines()

    # Diccionario para actualizar manifest
    sav_to_xlsx = {}

    for root, dirs, files in os.walk(RAW_BASE):
        for f in files:
            if f.endswith(".sav"):
                inputFile = os.path.join(root, f)
                outputFile = inputFile.replace(".sav", ".xlsx")
                
                # Convertir
                convertSavToXlsx(inputFile, outputFile)

                # Borrar .sav
                os.remove(inputFile)
                print(f"\tEliminado: {inputFile}")

                # Guardar correspondencia
                sav_to_xlsx[f] = os.path.basename(outputFile)

    # Actualizar manifest.log
    if sav_to_xlsx and manifest_lines:
        updated_lines = []
        for line in manifest_lines:
            for sav_name, xlsx_name in sav_to_xlsx.items():
                if sav_name in line:
                    line = line.replace(sav_name, xlsx_name)
            updated_lines.append(line)

        with open(manifest_path, "w", encoding="utf-8") as mf:
            mf.writelines(updated_lines)
        print(f"Manifest actualizado: {manifest_path}")

    # Crear cache
    with open(cacheFilePath, "w") as cacheFile:
        cacheFile.write("Transformación de .sav completada.\n")
    print(f"Caché creado: {cacheFilePath}")

### Separación de data raw y conversión a csv

In [10]:
def separateExcelToCsv():
    cacheFilePath = os.path.join(CACHE_BASE, CACHE_SEPARATE)
    if os.path.exists(cacheFilePath):
        print("Archivo de caché encontrado. Ya se ha ejecutado la separación de Excel a CSV.")
        return

    # Aseguramos que existan las carpetas
    os.makedirs(HECHOS_BASE, exist_ok=True)
    os.makedirs(VEHICULOS_BASE, exist_ok=True)
    os.makedirs(FL_BASE, exist_ok=True)

    if not os.path.exists(MANIFEST_BASE):
        print(f"No se encontró {MANIFEST_BASE}")
        return

    # Leer manifest.log original
    with open(MANIFEST_BASE, "r", encoding="utf-8") as mf:
        lines = mf.readlines()

    # Diccionarios para almacenar los nuevos manifest
    new_manifest = {
        "hechos": [],
        "vehiculos": [],
        "fallecidos_lesionados": []
    }

    # Expresiones regulares para identificar tipo y año
    pattern = re.compile(r"(\w+): .*? -> .*/(\d{4})/([^/]+\.xlsx)")

    for line in lines:
        match = pattern.search(line)
        if not match:
            continue
        tipo, year, xlsx_file = match.groups()

        # Determinar carpeta de destino
        if tipo == "hechos":
            base_path = HECHOS_BASE
        elif tipo == "vehiculos":
            base_path = VEHICULOS_BASE
        elif tipo == "fallecidos_lesionados":
            base_path = FL_BASE
        else:
            continue

        # Crear nombre de CSV: tipo_año.csv o tipo_año_num.csv si ya existe
        csv_name = f"{tipo}_{year}.csv"
        csv_path = os.path.join(base_path, csv_name)
        i = 1
        while os.path.exists(csv_path):
            csv_name = f"{tipo}_{year}_{i}.csv"
            csv_path = os.path.join(base_path, csv_name)
            i += 1

        # Leer Excel y guardar a CSV
        xlsx_path = os.path.join(RAW_BASE, year, xlsx_file)
        if os.path.exists(xlsx_path):
            df = pd.read_excel(xlsx_path)
            df.to_csv(csv_path, index=False, encoding="utf-8")
            print(f"Guardado CSV: {csv_path}")

            # Actualizar manifest del tipo
            new_manifest[tipo].append(f"{csv_name} -> {xlsx_file}\n")

    # Guardar manifest dentro de cada carpeta
    for tipo, entries in new_manifest.items():
        if entries:
            if tipo == "hechos":
                manifest_path = os.path.join(HECHOS_BASE, "manifest.log")
            elif tipo == "vehiculos":
                manifest_path = os.path.join(VEHICULOS_BASE, "manifest.log")
            else:
                manifest_path = os.path.join(FL_BASE, "manifest.log")
            
            with open(manifest_path, "w", encoding="utf-8") as mf:
                mf.writelines(entries)
            print(f"Manifest creado: {manifest_path}")

    # Crear cache
    with open(cacheFilePath, "w") as cacheFile:
        cacheFile.write("Separación de Excel a CSV completada.\n")
    print(f"Caché creado: {cacheFilePath}")

In [11]:
def organizeVariableFiles():
    """
    Mueve archivos Excel de las carpetas por año dentro de VARIABLE_BASE
    a subcarpetas por tipo (hechos, vehiculos, fallecidos_lesionados),
    los renombra como tipo_año.xlsx y genera un manifest.log con el mapeo
    nombre original -> nombre nuevo. Elimina carpetas de años si quedan vacías.
    """
    # Comprobar cache
    if os.path.exists(CACHE_SEPARATE_VAR):
        print("Archivo de caché encontrado. Separación de archivos de variables ya realizada.")
        return

    # Leer manifest
    if not os.path.exists(MANIFEST_BASE_VAR):
        print(f"No se encontró {MANIFEST_BASE_VAR}")
        return

    with open(MANIFEST_BASE_VAR, "r", encoding="utf-8") as mf:
        lines = mf.readlines()

    pattern = re.compile(r"(\w+): .*? -> .*/(\d{4})/([^/]+\.xls[x]?)")

    # Crear carpetas destino dentro de VARIABLE_BASE
    hechos_dir = os.path.join(VARIABLE_BASE, "hechos")
    vehiculos_dir = os.path.join(VARIABLE_BASE, "vehiculos")
    fl_dir = os.path.join(VARIABLE_BASE, "fallecidos_lesionados")

    os.makedirs(hechos_dir, exist_ok=True)
    os.makedirs(vehiculos_dir, exist_ok=True)
    os.makedirs(fl_dir, exist_ok=True)

    # Diccionarios para manifest
    manifests = {
        "hechos": [],
        "vehiculos": [],
        "fallecidos_lesionados": []
    }

    for line in lines:
        match = pattern.search(line)
        if not match:
            continue
        tipo, year, filename = match.groups()
        src_path = os.path.join(VARIABLE_BASE, year, filename)

        if tipo == "hechos":
            dest_dir = hechos_dir
        elif tipo == "vehiculos":
            dest_dir = vehiculos_dir
        elif tipo == "fallecidos_lesionados":
            dest_dir = fl_dir
        else:
            continue

        # Nuevo nombre: tipo_año.xlsx (o .xls)
        ext = os.path.splitext(filename)[1]
        dest_name = f"{tipo}_{year}{ext}"
        dest_path = os.path.join(dest_dir, dest_name)

        # Evitar sobrescribir
        i = 1
        while os.path.exists(dest_path):
            dest_name = f"{tipo}_{year}_{i}{ext}"
            dest_path = os.path.join(dest_dir, dest_name)
            i += 1

        shutil.move(src_path, dest_path)
        print(f"Movido: {src_path} -> {dest_path}")

        # Registrar en manifest
        manifests[tipo].append(f"{dest_name} -> {filename}\n")

    # Guardar manifest dentro de cada carpeta
    for tipo, entries in manifests.items():
        if entries:
            manifest_path = os.path.join(VARIABLE_BASE, tipo, "manifest.log")
            with open(manifest_path, "w", encoding="utf-8") as mf:
                mf.writelines(entries)
            print(f"Manifest creado: {manifest_path}")

    # Eliminar carpetas de años si quedaron vacías
    for year_dir in os.listdir(VARIABLE_BASE):
        full_path = os.path.join(VARIABLE_BASE, year_dir)
        if os.path.isdir(full_path) and year_dir not in ["hechos", "vehiculos", "fallecidos_lesionados"]:
            try:
                os.rmdir(full_path)  # solo elimina si está vacía
            except OSError:
                print(f"No se pudo eliminar {full_path}, no está vacía")

    # Crear cache
    with open(CACHE_SEPARATE_VAR, "w") as cf:
        cf.write("Archivos de variables movidos a carpetas por tipo dentro de VARIABLE_BASE.\n")
    print("Separación completada y cache creada.")


### Limpieza

In [12]:
# Comunes
MESES = {1:"enero",2:"febrero",3:"marzo",4:"abril",5:"mayo",6:"junio",7:"julio",8:"agosto",9:"septiembre",10:"octubre",11:"noviembre",12:"diciembre"}
DIAS  = {1:"lunes",2:"martes",3:"miercoles",4:"jueves",5:"viernes",6:"sabado",7:"domingo"}
FRANJA_HORA = {0:"null",1:"00:00 a 05:59",2:"06:00 a 11:59",3:"12:00 a 17:59",4:"18:00 a 23:59",99:"ignorado"}
RANGO_ANIO_DICT = {1:"1970-1979",2:"1980-1989",3:"1990-1999",4:"2000-2009",5:"2010-2019",6:"2020-2029",99:"ignorado"}
MAYOR_MENOR_MAPPING = {1:"mayor",2:"menor",9:"ignorado"}
SEXO_DICT = {1:"hombre",2:"mujer",9:"ignorado"}
MODEL_VEH_IGNORADO = 9999
MARCAS = {1:"ahm",2:"tvs apache",3:"asia hero",4:"audi",5:"avanti",6:"bajaj",7:"blue bird",8:"bmw",9:"changan",10:"chevrolet",11:"chrysler",12:"daewoo",13:"daihatsu",14:"datsun",15:"dodge",16:"fiat",17:"ford",18:"freedom",19:"freightliner",20:"genesis",21:"geo",22:"alfa romeo",23:"gmc",24:"great wall",25:"hero hunk",26:"hino",27:"honda",28:"hyundai",29:"infiniti",30:"internacional",31:"isuzu",32:"italika",33:"jaguar",34:"jeep",35:"jialing",36:"john deere",37:"jumbo",38:"kawasaki",39:"keeway",40:"kenworth",41:"kia",42:"kinlon",43:"kymco",44:"land rover",45:"loncin",46:"mahindra",47:"lexus",48:"mazda",49:"mci",50:"mercedes benz",51:"mini cooper",52:"mitsubishi",53:"motolansa",54:"movesa",55:"porche",56:"nissan",57:"peter bilt",58:"peugeot",59:"piaggio",60:"plymouth",61:"renault",62:"rhino",63:"sanyang",64:"serpento",65:"soprano",66:"subaru",67:"suzuki",68:"sym",69:"toyota",70:"cadillac",71:"harley davidson",72:"united motors (um)",73:"volkswagen",74:"volvo",75:"wuling",76:"yamaha",77:"yumbo dakar",78:"ferrari",79:"seat",80:"autoriksha",81:"eagle flight",83:"honlei",84:"jac",85:"jinbei",86:"hummer",87:"x-bow krm",88:"mack",89:"pontiac",90:"scion",91:"zx auto",92:"sterling",93:"yiben",94:"ritchie bros",95:"chery destiny",96:"vento",97:"citroen",98:"dayun",99:"jmc",100:"haojue",101:"hyosung",102:"jincheng",103:"lifan",104:"macat",105:"pegasus",106:"rolls royse",107:"acura",108:"kingo",109:"raybar",110:"regent",111:"sing",112:"shineray",113:"skigo",114:"tough",115:"wuyang",116:"caterpillar",117:"opel",118:"scrambler",119:"mercury",120:"saturn corporation",121:"byd",122:"lincoln",123:"shimano",124:"vecesa",125:"dayang",126:"dinamo",127:"dpv dover",128:"faw jiabao",129:"fisher",130:"frontier",131:"makiba",132:"motinsa",133:"kinroad",134:"qingqi",135:"transmetro",136:"wanfeng",137:"cadisa",138:"ssangyong",139:"daelim",140:"bashan",141:"zongshen",142:"sukida",143:"chongqing",144:"foton",145:"chana",146:"prymouth",147:"aprilia",148:"fastran",149:"skoda",150:"mustang",151:"geely",152:"daifo",153:"workman",154:"forward",155:"king long",156:"automosa",157:"gac gonow",158:"ud trucks",159:"scania",160:"new holland",161:"freeway",162:"keiko",163:"benelli",164:"jetix",165:"tmr sl",166:"ape city",167:"brigadier",168:"mrt",169:"oldsmobile",170:"ktm",171:"bmx",172:"senke",173:"cf moto",174:"busscar",175:"sinotruk",176:"sanlg",177:"jiangsu sinki",178:"berliet",179:"aurus senat",180:"fruehauf",181:"kioti",182:"maya tour",183:"sanya",184:"triumph",185:"kenton",186:"skygo",187:"great dane",188:"phenix",189:"segway",190:"otros",999:"ignorado"}

# Tipos de vehículo por año
tipoV_2013_2015 = {1:"automovil",2:"camioneta",3:"pick up",4:"motocicleta",5:"camion",6:"cabezal",7:"bus extraurbano",8:"jeep",9:"microbus",10:"taxi",11:"panel",12:"bus urbano",13:"tractor",14:"mototaxi",15:"furgon",16:"grua",17:"bus escolar",18:"bicicleta",99:"ignorado"}
tipoV_2016 = {1:"automovil",2:"camioneta sport o blazer",3:"pick up",4:"motocicleta",5:"camion",6:"cabezal",7:"bus extraurbano",8:"jeep",9:"microbus",10:"taxi",11:"panel",12:"bus urbano",13:"tractor",14:"mototaxi",15:"furgon",16:"grua",17:"bus escolar",18:"bicicleta",99:"ignorado"}
tipoV_2017 = {1:"automovil",2:"camioneta sport o blazer",3:"pick up",4:"motocicleta",5:"camion",6:"cabezal",7:"bus extraurbano",8:"jeep",9:"microbus",10:"taxi",11:"panel",12:"bus urbano",13:"tractor",14:"mototaxi",15:"furgon",16:"grua",17:"bus escolar",18:"bicicleta",19:"avioneta",20:"montacargas",21:"bus militar",22:"cuatrimoto",99:"ignorado"}
tipoV_2018_2020 = {1:"automovil",2:"camioneta sport o blazer",3:"pick up",4:"motocicleta",5:"camion",6:"cabezal",7:"bus extraurbano",8:"jeep",9:"microbus",10:"taxi",11:"panel",12:"bus urbano",13:"tractor",14:"mototaxi",15:"furgon",16:"grua",17:"bus escolar",18:"bicicleta",19:"avioneta",20:"montacargas",21:"bus militar",22:"cuatrimoto",23:"furgoneta",99:"ignorado"}
tipoV_2021 = {1:"automovil",2:"camioneta",3:"pick up",4:"motocicleta",5:"camion",6:"cabezal",7:"bus extraurbano",8:"jeep",9:"microbus",10:"taxi",11:"panel",12:"bus urbano",13:"tractor",14:"mototaxi",15:"furgon",16:"grua",17:"bus escolar",18:"bicicleta",19:"avioneta",20:"montacargas",21:"bus militar",22:"cuatrimoto",23:"furgoneta",99:"ignorado"}

# Tipos de accidente por año
accidentes_2013 = {1:"colision",2:"choque",3:"vuelco",4:"caida",11:"atropello",99:"ignorado"}
accidentes_2014_2017 = {1:"colision",2:"choque",3:"vuelco",4:"caida",5:"atropello",6:"perdida de control",7:"colision contra animal",8:"exceso de pasaje",9:"asfalto mojado",10:"exceso de velocidad",11:"desperfectos mecanicos",12:"incendio",99:"ignorado"}
accidentes_2018_2020 = {1:"colision",2:"choque",3:"vuelco",4:"caida",5:"atropello",6:"derrape",7:"embarranco",8:"encuneto",99:"ignorado"}
accidentes_2021 = {1:"colision",2:"choque",3:"vuelco",4:"caida",5:"atropello",6:"derrape",7:"desprendimiento",8:"incendio",9:"ataque armado",99:"ignorado"}

# Grupos de edad
g_edad_2013 = {1:"menor de 15",2:"15-19",3:"20-24",4:"25-29",5:"30-34",6:"35-39",7:"40-44",8:"45-49",9:"50-54",10:"55 y mas"}
g_edad_2014 = {1:"menor de 15",2:"15-19",3:"20-24",4:"25-29",5:"30-34",6:"35-39",7:"40-44",8:"45-49",9:"50-54",10:"55-59",11:"60 y mas",12:"ignorado"}
g_edad_60ymas = {1:"menor de 15",2:"15-19",3:"20-24",4:"25-29",5:"30-34",6:"35-39",7:"40-44",8:"45-49",9:"50-54",10:"55-59",11:"60 y mas",12:"ignorado"}
g_edad_80ymas = {1:"menor de 15",2:"15-19",3:"20-24",4:"25-29",5:"30-34",6:"35-39",7:"40-44",8:"45-49",9:"50-54",10:"55-59",11:"60-64",12:"65-69",13:"70-74",14:"75-80",15:"80 y mas",16:"ignorado"}

# Mapa de edades simplificado (compacto)
edad_map = {2013:{"g_edad":g_edad_2013}, 2014:{"g_edad":g_edad_2014}}
for _anio in range(2015, 2022):
    edad_map[_anio] = {"g_edad_60ymas":g_edad_60ymas,"g_edad_80ymas":g_edad_80ymas}

# Estado del conductor (2013-2017 y 2018-2021) — nombres mantenidos tal cual
estadoCond_2013_2017 = {1:"no ebrio",2:"ebrio",3:"drogado",9:"ignorado"}
estadoCond_2018_2021 = {1:"no ebrio",2:"ebrio",9:"ignorado"}

# Colores por año (renombrados)
color_2013_2017 = {
    1:"rojo", 2:"blanco", 3:"azul", 4:"gris", 5:"negro",
    6:"verde", 7:"amarillo", 8:"celeste", 9:"corinto", 10:"cafe",
    11:"beige", 12:"turquesa", 13:"marfil", 14:"anaranjado",
    15:"aqua", 16:"morado", 17:"rosado", 99:"ignorado"
}

color_2018_2021 = {
    1:"rojo", 2:"blanco", 3:"azul", 4:"gris", 5:"negro",
    6:"verde", 7:"amarillo", 8:"celeste", 9:"corinto", 10:"cafe",
    11:"beige", 12:"turquesa", 13:"marfil", 14:"anaranjado",
    15:"morado", 16:"rosado", 17:"varios colores", 99:"ignorado"
}

In [13]:
def normalize_text(text):
    """Normaliza texto: minúsculas, sin tildes, sin espacios extra"""
    if pd.isna(text):
        return ""
    return unidecode.unidecode(str(text)).lower().strip()


def extract_year_from_filename(filename):
    """Extrae el año del archivo, asumiendo formato *_YYYY.csv"""
    match = re.search(r'(\d{4})', filename)
    return int(match.group(1)) if match else None

def normalize_mes_dia(value, tipo="mes", year=None):
    if year is None or pd.isna(value):
        return pd.NA
    if tipo == "mes":
        return MESES.get(int(value), pd.NA) if 2013 <= year <= 2021 else normalize_text(value)
    elif tipo == "dia":
        return DIAS.get(int(value), pd.NA) if 2013 <= year <= 2021 else normalize_text(value)
    return value


def normalize_g_hora(value, year=None):
    if pd.isna(value) or value == "":
        return 0
    if year is None:
        return value
    if 2013 <= year <= 2021:
        try:
            return int(value)
        except:
            return 0
    elif 2022 <= year <= 2023:
        for k, v in FRANJA_HORA.items():
            if str(value).strip().lower() == v.lower():
                return k
        return 0
    return value


def normalize_zona(value):
    if pd.isna(value):
        return pd.NA
    v = normalize_text(value)
    if v in ["ignorada", "ignorado"]:
        return 99
    try:
        return int(v)
    except:
        return pd.NA

def normalize_anio_ocu(df):
    """
    - anio_ocu_dataset: copia de lo que había en 'ano_ocu' original
    - anio_ocu: si existe 'ano_ocu', se usa; si no, se usa 'anio_ocu_file'
    """
    if "ano_ocu" in df.columns:
        df["anio_ocu_dataset"] = df["ano_ocu"]
        df["anio_ocu"] = df["ano_ocu"]
    else:
        df["anio_ocu_dataset"] = pd.NA
        df["anio_ocu"] = pd.NA

    if "anio_ocu_file" in df.columns:
        df["anio_ocu"] = df["anio_ocu"].fillna(df["anio_ocu_file"])

    if "ano_ocu" in df.columns:
        df.drop(columns=["ano_ocu"], inplace=True)

    return df

def map_depto_mupio(df, path_deptos="data/departamentos.csv", path_mupios="data/municipios.csv"):
    # Cargar catálogos
    df_deptos = pd.read_csv(path_deptos, encoding="utf-8")
    df_mupios = pd.read_csv(path_mupios, encoding="utf-8")

    # Normalizar catálogos una sola vez
    depto_map = {normalize_text(row["departamento"]): row["codigo"] for _, row in df_deptos.iterrows()}
    muni_map = {normalize_text(row["municipio"]): row["codigo"] for _, row in df_mupios.iterrows()}

    def match_partial(val, catalog):
        if pd.isna(val):
            return pd.NA
        try:
            return int(val)
        except:
            val_norm = normalize_text(val)
            if "?" in val_norm:
                idx = val_norm.index("?")
                part_len = max(idx, 3)
                part_val = val_norm[:part_len]
            else:
                part_val = val_norm

            for key_norm, code in catalog.items():
                cmp_len = min(len(part_val), len(key_norm))
                if part_val[:cmp_len] == key_norm[:cmp_len]:
                    return code
            return pd.NA

    df["depto_ocu"] = df["depto_ocu"].apply(lambda x: match_partial(x, depto_map))
    df["mupio_ocu"] = df["mupio_ocu"].apply(lambda x: match_partial(x, muni_map))

    df["depto_ocu"] = pd.to_numeric(df["depto_ocu"], errors="coerce").astype("Int64")
    df["mupio_ocu"] = pd.to_numeric(df["mupio_ocu"], errors="coerce").astype("Int64")

    return df

def map_accidente_year(tipo_eve, anio):
    """
    Convierte el valor de tipo_eve a su nombre correspondiente según el año.
    Si el año es 2022 o 2023, solo normaliza a minúsculas y quita tildes.
    """


    # Si es 2022 o 2023: minúsculas y quitar tildes
    if anio in [2022, 2023]:
        val = str(tipo_eve).lower().strip()
        val = val.replace("colisi?n", "colision")
        val = unidecode.unidecode(val)  # <--- quita todas las tildes
        return val

    # Convertir números a nombre según año
    try:
        tipo_eve_int = int(tipo_eve)
    except:
        return str(tipo_eve).lower().strip()

    if anio == 2013:
        return accidentes_2013.get(tipo_eve_int, str(tipo_eve).lower().strip())
    elif anio in [2014, 2015, 2016, 2017]:
        return accidentes_2014_2017.get(tipo_eve_int, str(tipo_eve).lower().strip())
    elif anio in [2018, 2019, 2020]:
        return accidentes_2018_2020.get(tipo_eve_int, str(tipo_eve).lower().strip())
    elif anio == 2021:
        return accidentes_2021.get(tipo_eve_int, str(tipo_eve).lower().strip())
    else:
        return str(tipo_eve).lower().strip()

def map_tipo_veh_year(tipo_veh, anio):
    """
    Normaliza el tipo de vehículo según el año.
    Para 2022 y 2023: minúsculas, quita tildes y corrige caracteres.
    Para años anteriores: convierte número a string según diccionarios.
    """
    if pd.isna(tipo_veh):
        return pd.NA

    # Si es 2022 o 2023: minúsculas, quita tildes y corrige caracteres
    if anio in [2022, 2023]:
        val = str(tipo_veh).lower().strip()
        val = unidecode.unidecode(val)
        reemplazos = {
            "autom?vil": "automovil",
            "cami?n": "camion",
            "microb?s": "microbus",
            "furg?n": "furgon",
            "gr?a": "grua",
            "motos acua?ticas": "motos acuaticas"
        }
        for k, v in reemplazos.items():
            if k in val:
                val = val.replace(k, v)
        return val

    # Para años anteriores: convertir números a string según diccionario
    try:
        tipo_veh_int = int(tipo_veh)
    except:
        return str(tipo_veh).lower().strip()

    if anio in [2013, 2014, 2015]:
        return tipoV_2013_2015.get(tipo_veh_int, str(tipo_veh).lower().strip())
    elif anio == 2016:
        return tipoV_2016.get(tipo_veh_int, str(tipo_veh).lower().strip())
    elif anio == 2017:
        return tipoV_2017.get(tipo_veh_int, str(tipo_veh).lower().strip())
    elif anio in [2018, 2019, 2020]:
        return tipoV_2018_2020.get(tipo_veh_int, str(tipo_veh).lower().strip())
    elif anio == 2021:
        return tipoV_2021.get(tipo_veh_int, str(tipo_veh).lower().strip())
    else:
        return str(tipo_veh).lower().strip()

def map_color_veh_year(color_veh, anio):
    """
    Normaliza el color del vehículo según el año.
    Para 2022 y 2023: minúsculas, quita tildes y corrige caracteres.
    Para años anteriores: convierte número a string según diccionario.
    """
    if pd.isna(color_veh):
        return pd.NA


    # Si es 2022 o 2023: minúsculas, quitar tildes y corregir caracteres
    if anio in [2022, 2023]:
        val = str(color_veh).lower().strip()
        val = unidecode.unidecode(val)  # quitar tildes
        reemplazos = {
            "caf?e": "cafe"
        }
        for k, v in reemplazos.items():
            if k in val:
                val = val.replace(k, v)
        return val

    # Para años anteriores: convertir números a string según diccionario
    try:
        color_int = int(color_veh)
    except:
        return str(color_veh).lower().strip()

    if anio in [2013, 2014, 2015, 2016, 2017]:
        return color_2013_2017.get(color_int, str(color_veh).lower().strip())
    elif anio in [2018, 2019, 2020, 2021]:
        return color_2018_2021.get(color_int, str(color_veh).lower().strip())
    else:
        return str(color_veh).lower().strip()

def map_modelo_veh_year(modelo_veh, anio):
    """
    Normaliza el modelo del vehículo según el año.
    Para 2022 y 2023:
        - Si el valor es texto y dice 'ignorado', asigna la constante MODEL_VEH_IGNORADO.
        - Si es numérico, lo devuelve como int.
        - Si no se puede interpretar, devuelve pd.NA.
    Para otros años, devuelve el valor tal cual (conversión a int si aplica).
    """
    if pd.isna(modelo_veh):
        return pd.NA

    if anio in [2022, 2023]:
        val = str(modelo_veh).lower().strip()
        val = unidecode.unidecode(val)

        if val == "ignorado":
            return MODEL_VEH_IGNORADO

        try:
            return int(val)
        except:
            return pd.NA

    # Para otros años: intentar convertir a int, si no devolver string normalizado
    try:
        return int(modelo_veh)
    except:
        return str(modelo_veh).lower().strip()

def map_marca_veh_year(marca_veh, anio):
    """
    Normaliza la marca del vehículo según el año.
    - Para 2022 y 2023: pasa a minúsculas y quita tildes.
    - Para 2013-2021: mapea el valor numérico al diccionario de MARCAS.
      Si es NaN → se coloca 190 ("otros").
    """
    if pd.isna(marca_veh):
        # Si es NaN en 2013-2021 → "otros"
        if anio in range(2013, 2022):
            return MARCAS[190]
        return pd.NA

    # 2022 o 2023 → minúsculas y sin tildes
    if anio in [2022, 2023]:
        val = str(marca_veh).lower().strip()
        return unidecode.unidecode(val)

    # 2013-2021 → convertir a entero y mapear
    try:
        marca_int = int(marca_veh)
    except:
        return MARCAS[190]  # si no se puede convertir → "otros"

    return MARCAS.get(marca_int, MARCAS[190])

def map_estado_con_year(estado, anio):
    """
    Normaliza el estado del conductor según el año.
    - 2022-2023: solo minúsculas y quitar tildes.
    - 2013-2017: convertir número y mapear al diccionario correspondiente.
    - 2018-2021: convertir número y mapear al diccionario correspondiente.
    """
    if pd.isna(estado):
        return pd.NA

    if anio in [2022, 2023]:
        val = str(estado).lower().strip()
        return unidecode.unidecode(val)

    try:
        val_int = int(estado)
    except:
        return str(estado).lower().strip()

    if anio in [2013, 2014, 2015, 2016, 2017]:
        return estadoCond_2013_2017.get(val_int, "ignorado")
    elif anio in [2018, 2019, 2020, 2021]:
        return estadoCond_2018_2021.get(val_int, "ignorado")
    else:
        return str(estado).lower().strip()

def map_grupo_anio_veh(value, anio):
    """
    Convierte el valor de grupo_anio_veh según el año.
    Para 2013-2021: valores numericos se mapean a texto y se ajusta ignorado.
    Para 2022-2023: texto se normaliza y se convierte a numero si es posible.
    """
    # Si el valor es NaN
    if pd.isna(value):
        return "ignorado"

    # Convertir a texto para manejar todo uniforme
    val_str = str(value).strip()

    # 2013-2017
    if anio in [2013, 2014, 2015, 2016, 2017]:
        if val_str in ["6", "9999"]:
            return "ignorado"
        try:
            return RANGO_ANIO_DICT.get(int(float(val_str)), "ignorado")
        except:
            return "ignorado"
    
    # 2018-2020
    elif anio in [2018, 2019, 2020]:
        if val_str == "99":
            return "ignorado"
        try:
            return RANGO_ANIO_DICT.get(int(float(val_str)), "ignorado")
        except:
            return "ignorado"
    
    # 2021
    elif anio == 2021:
        if val_str == "99":
            return "ignorado"
        try:
            return RANGO_ANIO_DICT.get(int(float(val_str)), "ignorado")
        except:
            return "ignorado"

    # 2022-2023
    elif anio in [2022, 2023]:
        val_clean = unidecode.unidecode(val_str.lower())
        for k, v in RANGO_ANIO_DICT.items():
            if val_clean == v.lower():
                return v  # Retornar texto estandarizado
        if val_clean == "ignorado":
            return "ignorado"
        return val_clean  # cualquier otro texto se devuelve normalizado

    else:
        return "ignorado"

def map_mayor_menor(valor, anio):
    if pd.isna(valor):
        return "ignorado"
    if anio in [2022, 2023]:
        # Texto: pasar a minúsculas y quitar tildes
        if isinstance(valor, str):
            return valor.lower().replace("á","a").replace("é","e") \
                                .replace("í","i").replace("ó","o").replace("ú","u")
        return str(valor)
    else:
        # 2013-2021: valores numéricos
        try:
            valor_int = int(valor)
            return MAYOR_MENOR_MAPPING.get(valor_int, "ignorado")
        except:
            return "ignorado"

def map_sexo(value, anio):

    # Si anio es NaN o None, retornar ignorado
    if anio is None or pd.isna(anio):
        return "ignorado"

    # Asegurar que anio sea entero
    anio = int(anio)

    # Si valor es NaN o vacio
    if pd.isna(value) or value == "":
        return "ignorado"

    # Para 2013-2021: convertir a entero y mapear
    if 2013 <= anio <= 2021:
        try:
            val_int = int(float(value))  # convierte 1.0 a 1
            return SEXO_DICT.get(val_int, "ignorado")
        except:
            return "ignorado"

    # Para 2022-2023: texto a minusculas y sin tildes
    elif 2022 <= anio <= 2023:
        val_str = str(value).strip().lower()
        val_str = unidecode.unidecode(val_str)
        if val_str in ["hombre", "mujer"]:
            return val_str
        return "ignorado"

    else:
        return "ignorado"

# Función para mapear edades
def map_edad(value, anio, df):
    if pd.isna(value):
        return "ignorado"
    
    # 2022-2023 ya vienen como texto, solo normalizamos
    if anio >= 2022:
        return unidecode.unidecode(str(value).lower())
    
    # Para 2013-2021 buscamos en el diccionario correspondiente
    if anio in edad_map:
        col = None
        if anio in [2013, 2014]:
            col = "g_edad"
        else:
            if "g_edad_80ymas" in df.columns and pd.notna(df["g_edad_80ymas"].iloc[0]):
                col = "g_edad_80ymas"
            elif "g_edad_60ymas" in df.columns:
                col = "g_edad_60ymas"
        if col and col in edad_map[anio]:
            try:
                return unidecode.unidecode(str(edad_map[anio][col].get(int(value), "Ignorado")).lower())
            except:
                return "ignorado"
    return "ignorado"

### Generación de dataset a utilizar (hechos, vehiculos, fl)

In [14]:
def combineHechosCsv(GENERATED_LOG=False):
    if os.path.exists(HECHOS_CSV):
        os.remove(HECHOS_CSV)
        print(f"Archivo existente {HECHOS_CSV} eliminado para crear un nuevo combinado.")
        
    log_file = os.path.join(HECHOS_BASE, "hechos_combined.log")
    all_files = [f for f in os.listdir(HECHOS_BASE) if f.endswith(".csv")]
    if not all_files:
        print(f"No se encontraron CSV en {HECHOS_BASE}")
        return

    combined_df = pd.DataFrame()
    total_rows = 0
    last_num_corre = 0

    for file in all_files:
        file_path = os.path.join(HECHOS_BASE, file)
        df = pd.read_csv(file_path, low_memory=False)

        # Normalizar columnas
        df.columns = [col.lower()
                      .replace("á", "a").replace("é", "e").replace("í", "i")
                      .replace("ó", "o").replace("ú", "u").replace("ñ", "n")
                      .strip() for col in df.columns]

        # Eliminar columnas innecesarias
        drop_cols_initial = [
            "corre_base", "dia_ocu", "edad_quinquenales", "zona_ciudad",
            "areag_ocu", "area_geo_ocu", "sexo", "sexo_con", "sexo_per", "sexo_pil",
            "departamento", "municipio", "edad_pil", "g_edad_2", "edad_con", 
            "g_edad", "edad_per", "g_edad_80ymas", "g_edad_60ymas", "mayor_menor",
            "g_modelo_veh", "estado_con", "estado_pil"
        ]
        df.drop(columns=[c for c in drop_cols_initial if c in df.columns], inplace=True)

        # Año desde archivo
        df["anio_ocu_file"] = extract_year_from_filename(file)

        # Unificar causa_acc y tipo_eve
        if "causa_acc" in df.columns and "tipo_eve" in df.columns:
            df["tipo_eve"] = df["tipo_eve"].combine_first(df["causa_acc"])
            df.drop(columns=["causa_acc"], inplace=True)
        elif "causa_acc" in df.columns:
            df["tipo_eve"] = df["causa_acc"]
            df.drop(columns=["causa_acc"], inplace=True)

        # Normalizar tipo_eve y otros campos según año
        if "tipo_eve" in df.columns:
            df["tipo_eve"] = df.apply(lambda x: map_accidente_year(x["tipo_eve"], x["anio_ocu_file"]), axis=1)
        if "tipo_veh" in df.columns:
            df["tipo_veh"] = df.apply(lambda x: map_tipo_veh_year(x["tipo_veh"], x["anio_ocu_file"]), axis=1)
        if "color_veh" in df.columns:
            df["color_veh"] = df.apply(lambda x: map_color_veh_year(x["color_veh"], x["anio_ocu_file"]), axis=1)
        if "modelo_veh" in df.columns:
            df["modelo_veh"] = df.apply(lambda x: map_modelo_veh_year(x["modelo_veh"], x["anio_ocu_file"]), axis=1)
        if "marca_veh" in df.columns:
            df["marca_veh"] = df.apply(lambda x: map_marca_veh_year(x["marca_veh"], x["anio_ocu_file"]), axis=1)
        if "mayor_menor" in df.columns:
            df["mayor_menor"] = df.apply(lambda x: map_mayor_menor(x["mayor_menor"], x["anio_ocu_file"]), axis=1)

        # Unificar correlativos
        num_cols = [c for c in ["num_hecho", "num_corre", "num_correlativo"] if c in df.columns]
        df["num_corre"] = df[num_cols].bfill(axis=1).iloc[:, 0] if num_cols else pd.NA

        # Mes y hora
        mes_cols = [c for c in ["mes_ocu"] if c in df.columns]
        df["mes_ocu"] = df[mes_cols].bfill(axis=1).iloc[:, 0] if mes_cols else pd.NA
        hora_cols = [c for c in ["g_hora", "g_hora_5"] if c in df.columns]
        df["g_hora"] = df[hora_cols].bfill(axis=1).iloc[:, 0] if hora_cols else pd.NA

        # Zona
        if "zona_ocu" in df.columns:
            df["zona_ocu"] = df["zona_ocu"].apply(normalize_zona)

        # Eliminar columnas originales duplicadas
        drop_cols = ["num_hecho", "num_correlativo", "g_hora_5"]
        df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
        df = df.loc[:, ~df.columns.duplicated()]

        # Normalizar mes, día, hora
        df["mes_ocu"] = df.apply(lambda x: normalize_mes_dia(x["mes_ocu"], tipo="mes", year=x["anio_ocu_file"]), axis=1)
        if "dia_sem_ocu" in df.columns:
            df["dia_sem_ocu"] = df.apply(lambda x: normalize_mes_dia(x["dia_sem_ocu"], tipo="dia", year=x["anio_ocu_file"]), axis=1)
        df["g_hora"] = df.apply(lambda x: normalize_g_hora(x["g_hora"], year=x["anio_ocu_file"]), axis=1)

        # Correlativo acumulativo
        df["num_corre"] = df["num_corre"].fillna(0) + last_num_corre
        last_num_corre = df["num_corre"].max()

        # Normalizar año
        df = normalize_anio_ocu(df)

        # Mapear depto/municipio a códigos
        df = map_depto_mupio(df)

        # Concatenar al combinado
        combined_df = pd.concat([combined_df, df], axis=0, ignore_index=True, sort=False)
        print(f"{file}: {len(df)} filas")
        total_rows += len(df)

    print(f"Suma total de filas: {total_rows}")

    # Guardar combinado
    combined_df.to_csv(HECHOS_CSV, index=False)
    print(f"CSV combinado creado: {HECHOS_CSV}, tamaño final: {combined_df.shape}")

    # Log de muestra solo si GENERATED_LOG es True
    if GENERATED_LOG:
        with open(log_file, "w", encoding="utf-8") as f:
            f.write(f"Suma total de filas: {total_rows}\n\n")
            for year in sorted(combined_df["anio_ocu_file"].dropna().unique()):
                f.write(f"=== Año: {int(year)} ===\n")
                sample_rows = combined_df[combined_df["anio_ocu_file"] == year].head(2)
                f.write(sample_rows.to_string(index=False))
                f.write("\n\n")
        print(f"Log de muestra guardado en: {log_file}")

In [15]:
def combineVehiculosCsv(GENERATED_LOG=False):
    if os.path.exists(VEHICULOS_CSV):
        os.remove(VEHICULOS_CSV)
        print(f"Archivo existente {VEHICULOS_CSV} eliminado para crear un nuevo combinado.")

    log_file = os.path.join(VEHICULOS_BASE, "vehiculos_combined.log")
    all_files = [f for f in os.listdir(VEHICULOS_BASE) if f.endswith(".csv")]
    if not all_files:
        print(f"No se encontraron CSV en {VEHICULOS_BASE}")
        return

    combined_df = pd.DataFrame()
    total_rows = 0
    last_num_corre = 0

    for file in all_files:
        file_path = os.path.join(VEHICULOS_BASE, file)
        df = pd.read_csv(file_path, low_memory=False)

        # Normalizar nombres de columnas
        df.columns = [col.lower()
                      .replace("á", "a").replace("é", "e").replace("í", "i")
                      .replace("ó", "o").replace("ú", "u").replace("ñ", "n")
                      .strip() for col in df.columns]

        # Eliminar columnas innecesarias
        drop_cols_initial = [
            "corre_base", "dia_ocu", "edad_quinquenales", "zona_ciudad",
            "areag_ocu", "area_geo_ocu", "departamento", "municipio"
        ]
        df.drop(columns=[c for c in drop_cols_initial if c in df.columns], inplace=True)

        # Año desde archivo
        df["anio_ocu_file"] = extract_year_from_filename(file)

        # Unificar causa_acc y tipo_eve
        if "causa_acc" in df.columns and "tipo_eve" in df.columns:
            df["tipo_eve"] = df["tipo_eve"].combine_first(df["causa_acc"])
            df.drop(columns=["causa_acc"], inplace=True)
        elif "causa_acc" in df.columns:
            df["tipo_eve"] = df["causa_acc"]
            df.drop(columns=["causa_acc"], inplace=True)

        # Unificar estado_pil en estado_con
        if "estado_pil" in df.columns:
            if df["anio_ocu_file"].iloc[0] == 2013:
                if "estado_con" not in df.columns:
                    df["estado_con"] = df["estado_pil"]
                else:
                    df["estado_con"] = df["estado_con"].combine_first(df["estado_pil"])
            df.drop(columns=["estado_pil"], inplace=True)

        # Unificar grupo_mode_veh en g_modelo_veh
        if "grupo_mode_veh" in df.columns:
            if df["anio_ocu_file"].iloc[0] in [2013, 2014]:
                if "g_modelo_veh" not in df.columns:
                    df["g_modelo_veh"] = df["grupo_mode_veh"]
                else:
                    df["g_modelo_veh"] = df["g_modelo_veh"].combine_first(df["grupo_mode_veh"])
            df.drop(columns=["grupo_mode_veh"], inplace=True)

        # Unificar columnas de sexo en sexo_per
        if any(c in df.columns for c in ["sexo_pil", "sexo_con", "sexo_per"]):
            if "sexo_per" not in df.columns:
                df["sexo_per"] = pd.NA
            if "sexo_pil" in df.columns:
                df["sexo_per"] = df["sexo_per"].combine_first(df["sexo_pil"])
            if "sexo_con" in df.columns:
                df["sexo_per"] = df["sexo_per"].combine_first(df["sexo_con"])
            df.drop(columns=[c for c in ["sexo_pil", "sexo_con"] if c in df.columns], inplace=True)

        # Unificar columnas de edad en edad_per
        if any(c in df.columns for c in ["edad_pil", "edad_con", "edad_per"]):
            if "edad_per" not in df.columns:
                df["edad_per"] = pd.NA
            if "edad_pil" in df.columns:
                df["edad_per"] = df["edad_per"].combine_first(df["edad_pil"])
            if "edad_con" in df.columns:
                df["edad_per"] = df["edad_per"].combine_first(df["edad_con"])
            def safe_int(x):
                try:
                    return int(float(x))
                except:
                    return -1
            df["edad_per"] = df["edad_per"].apply(safe_int)
            df.drop(columns=[c for c in ["edad_pil", "edad_con"] if c in df.columns], inplace=True)

        # Crear columna unificada 'edad_grupo'
        if any(c in df.columns for c in ["g_edad", "g_edad_60ymas", "g_edad_80ymas"]):
            df["edad_grupo"] = pd.NA
            for c in ["g_edad", "g_edad_60ymas", "g_edad_80ymas"]:
                if c in df.columns:
                    df["edad_grupo"] = df["edad_grupo"].combine_first(df[c])
            df["edad_grupo"] = df.apply(lambda x: map_edad(x["edad_grupo"], x["anio_ocu_file"], df), axis=1)
            df["edad_grupo"] = df["edad_grupo"].apply(lambda x: unidecode.unidecode(str(x).lower()))
            df.drop(columns=[c for c in ["g_edad", "g_edad_60ymas", "g_edad_80ymas"] if c in df.columns], inplace=True)

        # Normalizar columnas por año
        if "tipo_eve" in df.columns:
            df["tipo_eve"] = df.apply(lambda x: map_accidente_year(x["tipo_eve"], x["anio_ocu_file"]), axis=1)
        if "tipo_veh" in df.columns:
            df["tipo_veh"] = df.apply(lambda x: map_tipo_veh_year(x["tipo_veh"], x["anio_ocu_file"]), axis=1)
        if "color_veh" in df.columns:
            df["color_veh"] = df.apply(lambda x: map_color_veh_year(x["color_veh"], x["anio_ocu_file"]), axis=1)
        if "modelo_veh" in df.columns:
            df["modelo_veh"] = df.apply(lambda x: map_modelo_veh_year(x["modelo_veh"], x["anio_ocu_file"]), axis=1)
        if "marca_veh" in df.columns:
            df["marca_veh"] = df.apply(lambda x: map_marca_veh_year(x["marca_veh"], x["anio_ocu_file"]), axis=1)
        if "estado_con" in df.columns:
            df["estado_con"] = df.apply(lambda x: map_estado_con_year(x["estado_con"], x["anio_ocu_file"]), axis=1)
        if "g_modelo_veh" in df.columns:
            df["g_modelo_veh"] = df.apply(lambda x: map_grupo_anio_veh(x["g_modelo_veh"], x["anio_ocu_file"]), axis=1)
        if "sexo_per" in df.columns:
            df["sexo_per"] = df.apply(lambda x: map_sexo(x["sexo_per"], x["anio_ocu_file"]), axis=1)

        # Unificar correlativos
        num_cols = [c for c in ["num_hecho", "num_corre", "num_correlativo"] if c in df.columns]
        df["num_corre"] = df[num_cols].bfill(axis=1).iloc[:, 0] if num_cols else pd.NA

        # Mes, hora y zona
        mes_cols = [c for c in ["mes_ocu"] if c in df.columns]
        df["mes_ocu"] = df[mes_cols].bfill(axis=1).iloc[:, 0] if mes_cols else pd.NA
        hora_cols = [c for c in ["g_hora", "g_hora_5"] if c in df.columns]
        df["g_hora"] = df[hora_cols].bfill(axis=1).iloc[:, 0] if hora_cols else pd.NA
        if "zona_ocu" in df.columns:
            df["zona_ocu"] = df["zona_ocu"].apply(normalize_zona)

        # Eliminar duplicados de columnas
        drop_cols = ["num_hecho", "num_correlativo", "g_hora_5"]
        df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
        df = df.loc[:, ~df.columns.duplicated()]

        # Normalizar mes, día, hora
        df["mes_ocu"] = df.apply(lambda x: normalize_mes_dia(x["mes_ocu"], tipo="mes", year=x["anio_ocu_file"]), axis=1)
        if "dia_sem_ocu" in df.columns:
            df["dia_sem_ocu"] = df.apply(lambda x: normalize_mes_dia(x["dia_sem_ocu"], tipo="dia", year=x["anio_ocu_file"]), axis=1)
        df["g_hora"] = df.apply(lambda x: normalize_g_hora(x["g_hora"], year=x["anio_ocu_file"]), axis=1)

        # Correlativo acumulativo
        df["num_corre"] = df["num_corre"].fillna(0) + last_num_corre
        last_num_corre = df["num_corre"].max()

        # Normalizar año y mapear depto/municipio
        df = normalize_anio_ocu(df)
        df = map_depto_mupio(df)

        # Concatenar al combinado
        combined_df = pd.concat([combined_df, df], axis=0, ignore_index=True, sort=False)
        print(f"{file}: {len(df)} filas")
        total_rows += len(df)

    print(f"Suma total de filas: {total_rows}")

    # Guardar combinado
    combined_df.to_csv(VEHICULOS_CSV, index=False)
    print(f"CSV combinado creado: {VEHICULOS_CSV}, tamaño final: {combined_df.shape}")

    # Log de muestra opcional
    if GENERATED_LOG:
        with open(log_file, "w", encoding="utf-8") as f:
            f.write(f"Suma total de filas: {total_rows}\n\n")
            for year in sorted(combined_df["anio_ocu_file"].dropna().unique()):
                f.write(f"=== Año: {int(year)} ===\n")
                sample_rows = combined_df[combined_df["anio_ocu_file"] == year].head(2)
                f.write(sample_rows.to_string(index=False))
                f.write("\n\n")
        print(f"Log de muestra guardado en: {log_file}")

In [16]:
def combineFlCsv():
    log_file = os.path.join(FL_BASE, "fl_combined.log")
    all_files = [f for f in os.listdir(FL_BASE) if f.endswith(".csv")]
    if not all_files:
        print(f"No se encontraron CSV en {FL_BASE}")
        return

    combined_df = pd.DataFrame()
    total_rows = 0
    last_num_corre = 0

    for file in all_files:
        file_path = os.path.join(FL_BASE, file)
        df = pd.read_csv(file_path, low_memory=False)

        # Normalizar columnas
        df.columns = [col.lower()
                      .replace("á", "a").replace("é", "e").replace("í", "i")
                      .replace("ó", "o").replace("ú", "u").replace("ñ", "n")
                      .strip() for col in df.columns]

        # Eliminar columnas innecesarias (ajusta según FL)
        drop_cols_initial = [
            "corre_base", "dia_ocu", "edad_quinquenales", "zona_ciudad",
            "areag_ocu", "area_geo_ocu", "departamento", "municipio",
            "sexo_con", "edad_con", "edad_pil", "sexo_pil", "sexo_per"
        ]
        df.drop(columns=[c for c in drop_cols_initial if c in df.columns], inplace=True)

        # Año desde archivo
        df["anio_ocu_file"] = extract_year_from_filename(file)

        # Unificar correlativos
        num_cols = [c for c in ["num_hecho", "num_corre", "num_correlativo"] if c in df.columns]
        df["num_corre"] = df[num_cols].bfill(axis=1).iloc[:, 0] if num_cols else pd.NA

        # Unificar causa_acc y tipo_eve
        if "causa_acc" in df.columns and "tipo_eve" in df.columns:
            df["tipo_eve"] = df["tipo_eve"].combine_first(df["causa_acc"])
            df.drop(columns=["causa_acc"], inplace=True)
        elif "causa_acc" in df.columns:
            df["tipo_eve"] = df["causa_acc"]
            df.drop(columns=["causa_acc"], inplace=True)

        # Normalizar tipo_eve: si es 2022 o 2023 solo a minúsculas y quitar tildes
        if "tipo_eve" in df.columns:
            df["tipo_eve"] = df.apply(lambda x: map_accidente_year(x["tipo_eve"], x["anio_ocu_file"]), axis=1)
        
        # Mes
        mes_cols = [c for c in ["mes_ocu"] if c in df.columns]
        df["mes_ocu"] = df[mes_cols].bfill(axis=1).iloc[:, 0] if mes_cols else pd.NA

        # Hora
        hora_cols = [c for c in ["g_hora", "g_hora_5"] if c in df.columns]
        df["g_hora"] = df[hora_cols].bfill(axis=1).iloc[:, 0] if hora_cols else pd.NA

        # Zona
        if "zona_ocu" in df.columns:
            df["zona_ocu"] = df["zona_ocu"].apply(normalize_zona)

        # Eliminar duplicados
        drop_cols = ["num_hecho", "num_correlativo", "g_hora_5"]
        df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
        df = df.loc[:, ~df.columns.duplicated()]

        # Normalizar mes, día, hora
        df["mes_ocu"] = df.apply(lambda x: normalize_mes_dia(x["mes_ocu"], tipo="mes", year=x["anio_ocu_file"]), axis=1)
        if "dia_sem_ocu" in df.columns:
            df["dia_sem_ocu"] = df.apply(lambda x: normalize_mes_dia(x["dia_sem_ocu"], tipo="dia", year=x["anio_ocu_file"]), axis=1)
        df["g_hora"] = df.apply(lambda x: normalize_g_hora(x["g_hora"], year=x["anio_ocu_file"]), axis=1)

        # Correlativo acumulativo
        df["num_corre"] = df["num_corre"].fillna(0) + last_num_corre
        last_num_corre = df["num_corre"].max()

        # Normalizar año
        df = normalize_anio_ocu(df)

        # Mapear depto/municipio (si existen en FL)
        df = map_depto_mupio(df)

        # Concatenar al combinado
        combined_df = pd.concat([combined_df, df], axis=0, ignore_index=True, sort=False)
        print(f"{file}: {len(df)} filas")
        total_rows += len(df)

    print(f"Suma total de filas: {total_rows}")

    # Guardar combinado
    combined_df.to_csv(FL_CSV, index=False)
    print(f"CSV combinado creado: {FL_CSV}, tamaño final: {combined_df.shape}")

    # Guardar log con 2 primeros registros por año
    with open(log_file, "w", encoding="utf-8") as f:
        f.write(f"Suma total de filas: {total_rows}\n\n")
        for year in sorted(combined_df["anio_ocu_file"].dropna().unique()):
            f.write(f"=== Año: {int(year)} ===\n")
            sample_rows = combined_df[combined_df["anio_ocu_file"] == year].head(2)
            f.write(sample_rows.to_string(index=False))
            f.write("\n\n")
    print(f"Log de muestra guardado en: {log_file}")

In [17]:
# def runCombineCsv():
#     """
#     Ejecuta la combinación de CSV de HECHOS, VEHICULOS y FL si no existe el cache.
#     Genera un manifest.log en data/ con todos los prints y un archivo de cache.
#     """
#     cacheFilePath = os.path.join(CACHE_BASE, CACHE_GENERATED_CSV)
#     manifestPath = os.path.join(DATA_BASE, "manifest.log")

#     if os.path.exists(cacheFilePath):
#         print("Archivo de caché encontrado. Ya se ha ejecutado el proceso de combinación de CSV.")
#         return

#     # Abrir manifest.log para capturar todos los prints
#     with open(manifestPath, "w", encoding="utf-8") as f:
#         with redirect_stdout(f):
#             print("=== Combinando HECHOS CSV ===")
#             combineHechosCsv()
#             print("\n=== Combinando VEHICULOS CSV ===")
#             combineVehiculosCsv()
#             print("\n=== Combinando FALLECIDOS/LESIONADOS CSV ===")
#             combineFlCsv()
#             print("\nProceso de combinación completado.")

#     # Crear archivo de cache
#     with open(cacheFilePath, "w") as f:
#         f.write("Cache generado: combinación de CSV realizada\n")

#     print(f"Proceso finalizado. Manifest guardado en {manifestPath}")
#     print(f"Archivo de cache generado en {cacheFilePath}")

**Glosario de columnas unificadas:**

| Columna estándar     | Significado                                         | Posibles nombres originales                                                                    | Archivos donde aparece | Observaciones                                                                 |
| -------------------- | --------------------------------------------------- | ---------------------------------------------------------------------------------------------- | ---------------------- | ----------------------------------------------------------------------------- |
| **num_corre**        | Número de correlativo del hecho/registro            | `num_hecho`, `núm_corre`, `num_corre`, `num_correlativo`, `num_correlativo_base`, `corre_base` | Hechos, Vehículos, FL  | Se unificó todo a `num_corre`.                                                |
| **dia_ocu**          | Día de ocurrencia del hecho                         | `dia_ocu`, `día_ocu`                                                                           | Hechos, Vehículos, FL  | Normalizado quitando tildes.                                                  |
| **mes_ocu**          | Mes de ocurrencia                                   | `mes_ocu`, `mes_ocurr`                                                                         | Hechos, Vehículos, FL  | Queda como `mes_ocu`.                                                         |
| **dia_sem_ocu**      | Día de la semana de ocurrencia                      | `dia_sem_ocu`, `día_sem_ocu`                                                                   | Hechos, Vehículos, FL  | Se estandarizó quitando acentos.                                              |
| **hora_ocu**         | Hora de ocurrencia                                  | `hora_ocu`                                                                                     | Hechos, Vehículos, FL  | Sin cambios.                                                                  |
| **g_hora**           | Grupo horario (ej. madrugada, mañana, tarde, noche) | `g_hora`, `grupo_hora`, `g_hora_5`                                                             | Hechos, Vehículos, FL  | Unificado a `g_hora`. Variantes como `g_hora_5` no se conservaron.            |
| **depto_ocu**        | Departamento donde ocurrió el hecho                 | `depto_ocu`, `departamento`                                                                    | Hechos, Vehículos, FL  | Todo a `depto_ocu`.                                                           |
| **mupio_ocu**        | Municipio del hecho                                 | `mupio_ocu`, `municipio`                                                                       | Hechos, Vehículos, FL  | Unificado como `mupio_ocu`.                                                   |
| **zona_ocu**         | Zona de ocurrencia (urbana/rural)                   | `zona_ocu`, `zona`                                                                             | Hechos, Vehículos, FL  | Se estandariza como `zona_ocu`.                                               |
| **tipo_eve**         | Tipo de evento / causa del accidente                | `tipo_eve`, `causa_acc`                                                                        | Hechos, FL             | Importante: en 2013 era `causa_acc`. Se unificó a `tipo_eve`.                 |
| **tipo_veh**         | Tipo de vehículo involucrado                        | `tipo_veh`, `vehiculo`                                                                         | Vehículos, FL          | Conservado como `tipo_veh`.                                                   |
| **marca_veh**        | Marca del vehículo                                  | `marca_veh`, `marca`                                                                           | Vehículos, FL          | Se unificó a `marca_veh`.                                                     |
| **color_veh**        | Color del vehículo                                  | `color_veh`, `color`                                                                           | Vehículos, FL          | Queda como `color_veh`.                                                       |
| **modelo_veh**       | Modelo del vehículo                                 | `modelo_veh`, `modelo`                                                                         | Vehículos, FL          | Unificado a `modelo_veh`.                                                     |
| **anio_ocu_dataset** | Año de ocurrencia según lo que viene en el CSV      | `año_ocu`                                                                                      | Hechos, Vehículos, FL  | Solo aparece en datasets recientes (ej. 2023). Si no está, queda vacío (NaN). |
| **anio_ocu_file**    | Año de ocurrencia extraído del nombre del archivo   | *(no existía en CSV original)*                                                                 | Hechos, Vehículos, FL  | Se agregó manualmente al leer el nombre del archivo (ej. *_2013.csv*).        |

**Columnas descartadas:**

* **latitud / longitud**
* **cod_depto / cod_mupio**
* **victimas**
* **observaciones**
* **g_hora_5** y **g_modelo_veh**

### Pipeline de preparación de data

In [18]:
runWebScrapping()

Archivo de caché encontrado. Ya se ha ejecutado el proceso de web_scrapping.


In [19]:
runWebScrappingVariables()

Archivo de caché encontrado. Ya se ha ejecutado el proceso de web_scrapping de variables.


In [20]:
organizeVariableFiles()

Archivo de caché encontrado. Separación de archivos de variables ya realizada.


In [21]:
processSavFiles()

Archivo de caché encontrado. Ya se ha ejecutado la conversión de .sav a .xlsx.


In [22]:
runGbCsv()

Archivo de caché encontrado. Ya se ha ejecutado el proceso de gb_csv.


In [23]:
separateExcelToCsv()

Archivo de caché encontrado. Ya se ha ejecutado la separación de Excel a CSV.


In [24]:
combineHechosCsv(False)

Archivo existente /home/jovyan/work/data/hechos/hechos_combinado.csv eliminado para crear un nuevo combinado.
hechos_2013.csv: 6324 filas
hechos_2014.csv: 5651 filas
hechos_2015.csv: 6854 filas
hechos_2016.csv: 7964 filas
hechos_2017.csv: 5879 filas
hechos_2018.csv: 6395 filas
hechos_2019.csv: 7047 filas
hechos_2020.csv: 6350 filas
hechos_2021.csv: 8153 filas
hechos_2022.csv: 7924 filas
hechos_2023.csv: 8218 filas
Suma total de filas: 76759
CSV combinado creado: /home/jovyan/work/data/hechos/hechos_combinado.csv, tamaño final: (76759, 16)
Log de muestra guardado en: /home/jovyan/work/data/hechos/hechos_combined.log


In [25]:
combineVehiculosCsv(False)

Archivo existente /home/jovyan/work/data/vehiculos/vehiculos_combinado.csv eliminado para crear un nuevo combinado.
vehiculos_2013.csv: 6323 filas
vehiculos_2014.csv: 7904 filas
vehiculos_2015.csv: 9823 filas
vehiculos_2016.csv: 11618 filas
vehiculos_2017.csv: 8644 filas
vehiculos_2018.csv: 9514 filas
vehiculos_2019.csv: 10827 filas
vehiculos_2020.csv: 10103 filas
vehiculos_2021.csv: 12796 filas
vehiculos_2022.csv: 12239 filas
vehiculos_2023.csv: 12197 filas
Suma total de filas: 111988
CSV combinado creado: /home/jovyan/work/data/vehiculos/vehiculos_combinado.csv, tamaño final: (111988, 22)
Log de muestra guardado en: /home/jovyan/work/data/vehiculos/vehiculos_combined.log


In [26]:
# combineVehiculosCsv_debug()

In [27]:
# combineFlCsv()

In [28]:
# runCombineCsv()

In [29]:
# printTree(RAW_BASE)

## Carga de Datos y Análisis Exploratorio

### Cargar en Spark (Databricks)

In [30]:
# def initSpark(appName="lab8"):
#     return (
#         SparkSession.builder
#         .master("local[*]")
#         .appName(appName)
#         .config("spark.sql.session.timeZone", "America/Guatemala")
#         .config("spark.driver.memory", "6g")
#         .config("spark.executor.memory", "6g")
#         .config("spark.sql.shuffle.partitions", "8")
#         .config("spark.sql.files.maxPartitionBytes", "134217728")
#         .getOrCreate()
#     )

In [31]:
# def readUnifiedCsvs(hechosCsv, vehiculosCsv, flCsv, spark):
#     hechos    = spark.read.option("header", True).option("inferSchema", True).csv(hechosCsv)
#     vehiculos = spark.read.option("header", True).option("inferSchema", True).csv(vehiculosCsv)
#     fl        = spark.read.option("header", True).option("inferSchema", True).csv(flCsv)
#     return hechos, vehiculos, fl

In [32]:
# def unifyYearCol(df):
#     has_ds   = "anio_ocu_dataset" in df.columns
#     has_file = "anio_ocu_file"    in df.columns

#     if has_ds and has_file:
#         expr = F.coalesce(F.col("anio_ocu_dataset").cast("int"),
#                           F.col("anio_ocu_file").cast("int"))
#     elif has_ds:
#         expr = F.col("anio_ocu_dataset").cast("int")
#     elif has_file:
#         expr = F.col("anio_ocu_file").cast("int")
#     else:
#         raise ValueError("No existe ninguna columna de año en el DataFrame.")
#     return df.withColumn("anio_ocu", expr)

In [33]:
# def castAndFilterYears(df, start=2013, end=2023):
#     return df.filter(F.col("anio_ocu").between(start, end))

In [34]:
# spark = initSpark()

# hechos_raw, vehiculos_raw, fl_raw = readUnifiedCsvs(HECHOS_CSV, VEHICULOS_CSV, FL_CSV, spark)

# hechos    = castAndFilterYears(unifyYearCol(hechos_raw))
# vehiculos = castAndFilterYears(unifyYearCol(vehiculos_raw))
# fl        = castAndFilterYears(unifyYearCol(fl_raw))

# hechos.cache(); vehiculos.cache(); fl.cache()

# print("cols hechos:", hechos.columns)
# print("cols vehiculos:", vehiculos.columns)
# print("cols fl:", fl.columns)

In [35]:
# # Función: tomar N registros completos por año (estratificado)
# from pyspark.sql import functions as F
# from pyspark.sql.window import Window

# def takePerYear(df, yearCol="anio_ocu", n=2, seed=42):
#     df2 = df.where(F.col(yearCol).isNotNull())
#     w = Window.partitionBy(yearCol).orderBy(F.rand(seed))
#     return (df2
#             .withColumn("_rn", F.row_number().over(w))
#             .filter(F.col("_rn") <= n)
#             .drop("_rn")
#             .orderBy(F.col(yearCol).asc()))

# # USO
# muestra_hechos    = takePerYear(hechos,    yearCol="anio_ocu", n=2)
# muestra_vehiculos = takePerYear(vehiculos, yearCol="anio_ocu", n=2)
# muestra_fl        = takePerYear(fl,        yearCol="anio_ocu", n=2)

# muestra_hechos.show(100, truncate=False)
# muestra_vehiculos.show(100, truncate=False)
# muestra_fl.show(100, truncate=False)


### 1. Conteo, muestra y estadísticas descriptivas por dataset

In [36]:
# def showCountSampleAndStats(df, importantCols=None, n=5, title=None):
#     if title: print(f"\n== {title} ==")
#     cnt = df.count()
#     print("count:", cnt)
#     df.show(n, truncate=False)
#     # columnas a describir
#     default = ["anio_ocu","mes_ocu","dia_sem_ocu","hora_ocu",
#                "depto_ocu","mupio_ocu","zona_ocu","tipo_eve",
#                "tipo_veh","marca_veh","color_veh","modelo_veh"]
#     cols = [c for c in (importantCols or default) if c in df.columns]
#     if cols:
#         df.describe(cols).show()
#         df.summary("count","min","25%","50%","75%","max").select(cols).show()
#     return cnt

In [37]:
# showCountSampleAndStats(
#     hechos,
#     importantCols=["anio_ocu","mes_ocu","dia_sem_ocu","hora_ocu",
#                    "depto_ocu","mupio_ocu","zona_ocu","tipo_eve"],
#     n=5, title="hechos"
# )

In [38]:
# showCountSampleAndStats(
#     vehiculos,
#     importantCols=["anio_ocu","mes_ocu","hora_ocu","depto_ocu","mupio_ocu",
#                    "zona_ocu","tipo_veh","marca_veh","color_veh","modelo_veh"],
#     n=5, title="vehiculos"
# )

In [39]:
# showCountSampleAndStats(
#     fl,
#     importantCols=["anio_ocu","mes_ocu","dia_sem_ocu","hora_ocu",
#                    "depto_ocu","mupio_ocu","zona_ocu","tipo_veh","color_veh","modelo_veh"],
#     n=5, title="fallecidos_lesionados"
# )

### 2. Validación de años disponibles por dataset

In [40]:
# def getYearsPerSource(df, colDs="anio_ocu_dataset", colFile="anio_ocu_file"):
#     years = {}
#     if colDs in df.columns:
#         years["dataset"] = (df.select(F.col(colDs).cast("int").alias("anio"))
#                               .where(F.col(colDs).isNotNull())
#                               .distinct().orderBy("anio"))
#     if colFile in df.columns:
#         years["file"] = (df.select(F.col(colFile).cast("int").alias("anio"))
#                            .where(F.col(colFile).isNotNull())
#                            .distinct().orderBy("anio"))
#     return years

In [41]:
# def yearConsistencySummary(df, name="", colDs="anio_ocu_dataset", colFile="anio_ocu_file"):
#     # columnas seguras (si no existen, se reemplazan por NULL)
#     y_ds   = F.col(colDs).cast("int")   if colDs  in df.columns else F.lit(None).cast("int")
#     y_file = F.col(colFile).cast("int") if colFile in df.columns else F.lit(None).cast("int")
#     d = df.select(y_ds.alias("y_ds"), y_file.alias("y_file"))

#     agg = d.agg(
#         F.count(F.lit(1)).alias("total"),
#         F.sum(F.when(F.col("y_ds").isNotNull() & F.col("y_file").isNotNull(), 1).otherwise(0)).alias("both_present"),
#         F.sum(F.when(F.col("y_ds").isNotNull() & F.col("y_file").isNotNull() & (F.col("y_ds")==F.col("y_file")), 1).otherwise(0)).alias("match")
#     ).withColumn("mismatch", F.col("both_present") - F.col("match")) \
#      .withColumn("not_confirmable", F.col("total") - F.col("both_present")) \
#      .withColumn("both_present_pct", F.col("both_present")/F.col("total")) \
#      .withColumn("match_pct_of_both", F.when(F.col("both_present")>0, F.col("match")/F.col("both_present"))) \
#      .withColumn("mismatch_pct_of_both", F.when(F.col("both_present")>0, F.col("mismatch")/F.col("both_present"))) \
#      .withColumn("not_confirmable_pct_of_total", F.col("not_confirmable")/F.col("total")) \
#      .withColumn("dataset", F.lit(name)) \
#      .select("dataset","total","both_present","both_present_pct","match","match_pct_of_both",
#              "mismatch","mismatch_pct_of_both","not_confirmable","not_confirmable_pct_of_total")
#     return agg

In [42]:
# # USO — Hechos
# yrs = getYearsPerSource(hechos)
# print("Años HECHOS (dataset):");  yrs.get("dataset", spark.createDataFrame([], "anio int")).show()
# print("Años HECHOS (file):");     yrs.get("file", spark.createDataFrame([], "anio int")).show()
# yearConsistencySummary(hechos, name="hechos").show(truncate=False)

# # USO — Vehículos
# yrs = getYearsPerSource(vehiculos)
# print("Años VEHICULOS (dataset):"); yrs.get("dataset", spark.createDataFrame([], "anio int")).show()
# print("Años VEHICULOS (file):");    yrs.get("file", spark.createDataFrame([], "anio int")).show()
# yearConsistencySummary(vehiculos, name="vehiculos").show(truncate=False)

# # USO — Fallecidos/Lesionados
# yrs = getYearsPerSource(fl)
# print("Años FL (dataset):");  yrs.get("dataset", spark.createDataFrame([], "anio int")).show()
# print("Años FL (file):");     yrs.get("file", spark.createDataFrame([], "anio int")).show()
# yearConsistencySummary(fl, name="fallecidos_lesionados").show(truncate=False)

### 3. Valores distintos de “tipo de accidente”

### 4. Departamentos únicos en las bases

## Preguntas de análisis

### 5. Accidentes por año y departamento (gráfico de barras)

### 6. Día de la semana con más accidentes en 2023 (gráfico de columnas)

### 7. Distribución horaria de accidentes en el municipio de Guatemala (histograma)

### 8. Unión Hechos–Vehículos (llave compuesta) y total de registros combinados